# Base vs SFT vs RL (PPO) on Python docstring generation

This notebook compares three checkpoints on the same docstring task as the rest of the repo (`make_prompt` in `src/utils.py`): **given function source, generate the docstring body only** (no echoed `def`, no full function repeated).

| Model | Source |
|-------|--------|
| **Base** | Same hub id as SFT cold-start: `BASE_MODEL` in `src/config.py` (default `Qwen/Qwen2.5-3B-Instruct`; override with env `BASE_MODEL`) |
| **SFT** | `outputs/sft_policy` (supervised fine-tuning on `data/phase_1/sft_train.jsonl`) |
| **RL** | `outputs/ppo_policy` (PPO after SFT; optional if you ran Phase 6) |

Goal: the base LM may **ramble, repeat the prompt, or describe unrelated code**; **SFT** fits the assistant format; **RL** further aligns with the reward model trained on preferences. Full metrics: `python src/phase_7/evaluate.py`. Run the cells below—if SFT/RL folders are missing, those sections print a short note instead of failing.

In [9]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # load .env so HUGGING_FACE_HUB_TOKEN is available

# Run from repo root so "src" is on the path
sys.path.insert(0, str(Path.cwd() / "src"))
from config import policy_base_model_id, policy_from_pretrained_kwargs, tokenizer_pretrained_kwargs
from utils import make_prompt

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

REPO_ROOT = Path.cwd()
SFT_DIR = REPO_ROOT / "outputs" / "sft_policy"
PPO_DIR = REPO_ROOT / "outputs" / "ppo_policy"

device = "cuda" if torch.cuda.is_available() else "cpu"
_base_kw = policy_from_pretrained_kwargs()
_tok_kw = tokenizer_pretrained_kwargs()


def load_local_causal_lm(path: Path):
    tok = AutoTokenizer.from_pretrained(str(path), **_tok_kw)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(str(path), **_base_kw)
    if _base_kw.get("device_map"):
        pass  # model already placed
    else:
        m = m.to(device)
    m.eval()
    return tok, m


# Base (original LM — same hub id as SFT cold-start: policy_base_model_id() / BASE_MODEL)
base_name = policy_base_model_id()
base_tokenizer = AutoTokenizer.from_pretrained(base_name, **_tok_kw)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(base_name, **_base_kw)
if _base_kw.get("device_map"):
    pass
else:
    base_model = base_model.to(device)
base_model.eval()
print(f"Base model: {base_name}")

# SFT and RL checkpoints (same family as training pipeline)
sft_tokenizer, sft_model = None, None
if SFT_DIR.exists():
    sft_tokenizer, sft_model = load_local_causal_lm(SFT_DIR)
    print(f"Loaded SFT from {SFT_DIR}")
else:
    print(f"SFT checkpoint not found at {SFT_DIR} — run Phase 2 (train_sft) to compare.")

ppo_tokenizer, ppo_model = None, None
if PPO_DIR.exists():
    ppo_tokenizer, ppo_model = load_local_causal_lm(PPO_DIR)
    print(f"Loaded RL (PPO) from {PPO_DIR}")
else:
    print(f"PPO checkpoint not found at {PPO_DIR} — run Phase 6 (train_ppo) to compare.")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 6035.54it/s]


Base model: Qwen/Qwen2.5-1.5B


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 5703.30it/s]


Loaded SFT from /home/apilla01/git/rl-prompt-injection-defender/outputs/sft_policy


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 5620.23it/s]


Loaded RL (PPO) from /home/apilla01/git/rl-prompt-injection-defender/outputs/ppo_policy


Three small Python functions (same schema as `data/phase_1/sft_train.jsonl`: `input` = source, `reference` = gold docstring body for eyeball comparison):

In [11]:
examples = [
    {
        "input": """def process_numbers(nums: list[int]) -> dict:
    def _filter_even(n: int) -> bool:
        return n % 2 == 0

    evens = [n for n in nums if _filter_even(n)]
    odds = [n for n in nums if not _filter_even(n)]

    def _sum(lst: list[int]) -> int:
        total = 0
        for x in lst:
            total += x
        return total

    return {"evens": evens, "odds": odds, "sum_evens": _sum(evens), "sum_odds": _sum(odds)}""",
        "reference": (
            "Partition integers into evens and odds and return element-wise sums.\n\n"
            "Args:\n"
            "    nums: List of integers.\n\n"
            "Returns:\n"
            "    Dict with keys ``evens``, ``odds``, ``sum_evens``, and ``sum_odds``."
        ),
    },
    {
        "input": """def clamp(value: float, low: float, high: float) -> float:
    return max(low, min(high, value))""",
        "reference": (
            "Clamp ``value`` to the inclusive range ``[low, high]``.\n\n"
            "Args:\n"
            "    value: Number to clamp.\n"
            "    low: Lower bound.\n"
            "    high: Upper bound.\n\n"
            "Returns:\n"
            "    The clamped value."
        ),
    },
    {
        "input": """import asyncio

async def sleep_ms(ms: int) -> None:
    await asyncio.sleep(ms / 1000.0)""",
        "reference": (
            "Sleep asynchronously for ``ms`` milliseconds.\n\n"
            "Args:\n"
            "    ms: Duration in milliseconds.\n\n"
            "Returns:\n"
            "    None."
        ),
    },
]

In [12]:
# Match src/phase_7/evaluate.py (docstring generations); repetition_penalty reduces loops.
MAX_NEW_TOKENS = 192
GEN_TEMPERATURE = 0.3
# Longer than evaluate.py's 512 so demo functions are not truncated; shorten if you hit OOM.
PROMPT_MAX_LENGTH = 1024

from IPython.display import Markdown, display


def truncate_stutter(text: str) -> str:
    """Cut off when the model repeats the same non-empty line (common small-LM failure mode)."""
    lines = text.splitlines()
    out = []
    for i, line in enumerate(lines):
        if i > 0 and line.strip() and line.strip() == lines[i - 1].strip():
            break
        out.append(line)
    return "\n".join(out).strip()


def generate_completion(
    tok,
    m,
    prompt: str,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = GEN_TEMPERATURE,
) -> str:
    """Decode only new tokens (same idea as src/phase_7/evaluate.py)."""
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=PROMPT_MAX_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        gen = m.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            repetition_penalty=1.12,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
    input_ids = inputs["input_ids"][0]
    new_tokens = gen[0, input_ids.shape[0] :]
    raw = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return truncate_stutter(raw)


for i, ex in enumerate(examples):
    user_input = ex["input"]
    reference = ex["reference"]
    prompt = make_prompt(user_input)
    print("=" * 60)
    display(
        Markdown(
            "**Function source** ("
            + str(i + 1)
            + "/"
            + str(len(examples))
            + ")\n\n```python\n"
            + user_input
            + "\n```"
        )
    )
    display(Markdown("**Reference (gold) docstring**\n\n```\n" + reference + "\n```"))
    print()
    print(f"--- Base ({base_name}) ---")
    print(generate_completion(base_tokenizer, base_model, prompt))
    print()
    if sft_model is not None:
        print("--- SFT (outputs/sft_policy) ---")
        print(generate_completion(sft_tokenizer, sft_model, prompt))
    else:
        print("--- SFT ---\n(skipped: no checkpoint)")
    print()
    if ppo_model is not None:
        print("--- RL / PPO (outputs/ppo_policy) ---")
        print(generate_completion(ppo_tokenizer, ppo_model, prompt))
    else:
        print("--- RL / PPO ---\n(skipped: no checkpoint)")
    print()

**Function source** (1/3)

```python
def process_numbers(nums: list[int]) -> dict:
    def _filter_even(n: int) -> bool:
        return n % 2 == 0

    evens = [n for n in nums if _filter_even(n)]
    odds = [n for n in nums if not _filter_even(n)]

    def _sum(lst: list[int]) -> int:
        total = 0
        for x in lst:
            total += x
        return total

    return {"evens": evens, "odds": odds, "sum_evens": _sum(evens), "sum_odds": _sum(odds)}
```

**Reference (gold) docstring**

```
Partition integers into evens and odds and return element-wise sums.

Args:
    nums: List of integers.

Returns:
    Dict with keys ``evens``, ``odds``, ``sum_evens``, and ``sum_odds``.
```


--- Base (Qwen/Qwen2.5-1.5B) ---
"""
This function takes a list of integers as input and returns a dictionary containing three lists: 'evens', 'odds', and their respective sums.
"""

--- SFT (outputs/sft_policy) ---
Process a list of integers into even/odd groups with summed totals.

Args:
    nums: A list of integer arguments to be processed.

Returns:
    A dictionary containing three keys:
        * ``evens`` – List of even numbers from the input.
        * ``odds`` – List of odd numbers from the input.
        * ``sum_evens`` – Sum of all even numbers.
        * ``sum_odds`` – Sum of all odd numbers.

Raises:
    None.

--- RL / PPO (outputs/ppo_policy) ---
Process a list of integers into even and odd numbers, compute their sums, and return a dictionary with these results.

Args:
    nums: A list of integer arguments to be processed.

Returns:
    A dictionary containing three keys:
        * ``evens`` – a list of even numbers from the input.
        * ``odds`` – a list of odd num

**Function source** (2/3)

```python
def clamp(value: float, low: float, high: float) -> float:
    return max(low, min(high, value))
```

**Reference (gold) docstring**

```
Clamp ``value`` to the inclusive range ``[low, high]``.

Args:
    value: Number to clamp.
    low: Lower bound.
    high: Upper bound.

Returns:
    The clamped value.
```


--- Base (Qwen/Qwen2.5-1.5B) ---
"""
This function takes a floating-point number as input and returns it clamped between two given bounds.
The lower bound is inclusive, while the upper bound is exclusive.
"""

--- SFT (outputs/sft_policy) ---
Clamp a numeric value to an inclusive range [low, high].

Args:
    value: The number to be clamped.
    low: Lower bound of the range; if ``value`` is less than this,
        it will be set to ``low``.
    high: Upper bound of the range; if ``value`` is greater than
        this, it will be set to ``high``.

Returns:
    A float between ``low`` and ``high``, inclusive.

Raises:
    None.

--- RL / PPO (outputs/ppo_policy) ---
Clamp a floating-point number between two limits.

Args:
    value: The numeric input to be clamped.
    low: Lower limit of the range; if ``value`` is less than this,
        it will be set to ``low``.
    high: Upper limit of the range; if ``value`` is greater than
        this, it will be set to ``high``.

Returns:
    A

**Function source** (3/3)

```python
import asyncio

async def sleep_ms(ms: int) -> None:
    await asyncio.sleep(ms / 1000.0)
```

**Reference (gold) docstring**

```
Sleep asynchronously for ``ms`` milliseconds.

Args:
    ms: Duration in milliseconds.

Returns:
    None.
```


--- Base (Qwen/Qwen2.5-1.5B) ---
"""
Sleeps for ms milliseconds.
"""

# The following is an example of how to use this function:

print(sleep_ms(500))

--- SFT (outputs/sft_policy) ---
Asynchronously waits for ``ms`` milliseconds using the platform's native sleep mechanism.

Args:
    ms: Number of milliseconds to wait.

Returns:
    Nothing; this coroutine does not return anything explicitly but may block
    execution until it completes.

--- RL / PPO (outputs/ppo_policy) ---
Asynchronously waits for ``ms`` milliseconds using the asyncio library.

Args:
    ms: Number of milliseconds to wait; must be non-negative.

Returns:
    Nothing.



## Takeaway

Compare generations to the **reference** docstrings: the base LM often **ignores the format**, repeats unrelated boilerplate, or continues with non-docstring text. **SFT** usually improves adherence and overlap with gold text; **PPO** shifts toward higher reward-model scores (watch for shorter or over-generic docstrings). Full metrics: `python src/phase_7/evaluate.py`.